# CSCI 4930: Group Project
**Mark Evers**  
**Marlon Williams**  
**John Sepulvuda**  
**Kyle Lowe**

---


## Install

In [1]:
# %pip install -r requirements.txt

## Imports

In [2]:
import gdown
import os
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import precision_recall_fscore_support, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import numpy as np

## Download dataset into workspace

In [3]:
# check if the file exists
if not os.path.exists("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv"):
    # create the data directory if it doesn't exist
    os.makedirs("data", exist_ok=True)
    # download the file from Google Drive
    gdown.download("https://drive.google.com/uc?id=1wcvSB8gTEAzbq2ARRhjGJGyoMlP-Rby3", "data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", quiet=False)

## Load the data

In [4]:
keep_cols = [
    "offense_code",
    # The two below are data leaks
    # "offense_code_extension",
    # "top_traffic_accident_offense",
    "first_occurrence_date",
    "district_id",
    "precinct_id",
    "neighborhood_id",
    "bicycle_ind",
    "pedestrian_ind",
    "HARMFUL_EVENT_SEQ_1",
    "HARMFUL_EVENT_SEQ_2",
    "HARMFUL_EVENT_SEQ_3",
    "road_location",
    "ROAD_DESCRIPTION",
    "ROAD_CONTOUR",
    "ROAD_CONDITION",
    "LIGHT_CONDITION",
    "TU1_VEHICLE_TYPE",
    "TU1_TRAVEL_DIRECTION",
    "TU1_VEHICLE_MOVEMENT",
    "TU1_DRIVER_ACTION",
    "TU1_DRIVER_HUMANCONTRIBFACTOR",
    "TU1_PEDESTRIAN_ACTION",
    "TU2_VEHICLE_TYPE",
    "TU2_TRAVEL_DIRECTION",
    "TU2_VEHICLE_MOVEMENT",
    "TU2_DRIVER_ACTION",
    "TU2_DRIVER_HUMANCONTRIBFACTOR",
    "TU2_PEDESTRIAN_ACTION",
    # "geo_lon",
    # "geo_lat",
    # these are the target variables
    "SERIOUSLY_INJURED",
    "FATALITIES"
]
df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", usecols=keep_cols, index_col=False)
print(f"Total rows: {len(df)}")
print(f"Index column name: {df.index.name}")

df["HARMFUL_EVENT_SEQ_2"].isna().sum()

Total rows: 282244
Index column name: None


/tmp/ipykernel_12461/3254251558.py:38: DtypeWarning: Columns (0: LIGHT_CONDITION) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", usecols=keep_cols, index_col=False)


np.int64(63727)

## Build the target column

In [5]:
print("Constructing high risk field...")

# Create a field where any fatality or serious injury
# is classified as "high risk".
df["high_risk"] = ((df["FATALITIES"].fillna(0) > 0) | (df["SERIOUSLY_INJURED"].fillna(0) > 0)).astype(int)
# drop the original target variables (data leak)
df.drop(columns=["SERIOUSLY_INJURED", "FATALITIES"], inplace=True)

print(f"Number of high risk accidents: {df['high_risk'].sum()}")

Constructing high risk field...
Number of high risk accidents: 6559


### Drop rows with missing values (currently not used)

In [6]:
# total_before = len(df)
# print(f"Total rows before dropping null values: {total_before}")

# n_before = len(df)
# df = df[(~df.SERIOUSLY_INJURED.isnull()) & (~df.FATALITIES.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column SERIOUSLY_INJURED and FATALITIES.")

# n_before = len(df)
# df = df[(~df.ROAD_DESCRIPTION.isnull()) & (~df.ROAD_CONTOUR.isnull()) & (~df.ROAD_CONDITION.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in columns ROAD_DESCRIPTION, ROAD_CONTOUR, and ROAD_CONDITION.")

# n_before = n_after
# df = df[(~df.HARMFUL_EVENT_SEQ_1.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column HARMFUL_EVENT_SEQ_1.")

# n_before = n_after
# df = df[(~df.TU1_TRAVEL_DIRECTION.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column TU1_TRAVEL_DIRECTION.")

# # n_before = n_after
# # df = df[(~df.geo_lon.isnull())]
# # n_after = len(df)
# # print(f"Dropped {n_before - n_after} rows with null values in column geo_lon and geo_lat.")

# n_before = n_after
# df = df[(~df.TU1_DRIVER_ACTION.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column TU1_DRIVER_ACTION.")

# n_before = n_after
# df = df[(~df.TU1_DRIVER_HUMANCONTRIBFACTOR.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column TU1_DRIVER_HUMANCONTRIBFACTOR.")

# total_after = len(df)
# print(f"Total rows after dropping null values: {total_after} ({total_before - total_after} rows dropped in total).")


## Fill in missing values and clean data

There was a lot of nonsense data with cells containing two blank spaces ("  ") or numbers as car types.  Most nonsense data was just one or two rows each.  We replace those cells with "None", which then later become a category in the one-hot encoded dataset.

In [7]:
print("Trimming whitespace...")
# df["top_traffic_accident_offense"] = df["top_traffic_accident_offense"].str.strip()
df["district_id"] = df["district_id"].str.strip()


print("Filling in missing values...")

# df["geo_lon"] = df["geo_lon"].fillna(0)
# df["geo_lat"] = df["geo_lat"].fillna(0)

df["district_id"] = df["district_id"].fillna("None")
df["precinct_id"] = df["precinct_id"].fillna("None")
df["neighborhood_id"] = df["neighborhood_id"].fillna("None")
df["bicycle_ind"] = df["bicycle_ind"].fillna(0)
df["pedestrian_ind"] = df["pedestrian_ind"].fillna(0)

df["HARMFUL_EVENT_SEQ_1"] = df["HARMFUL_EVENT_SEQ_1"].fillna("None").replace("  ", "None")
df["HARMFUL_EVENT_SEQ_2"] = df["HARMFUL_EVENT_SEQ_2"].fillna("None").replace("  ", "None")
df["HARMFUL_EVENT_SEQ_3"] = df["HARMFUL_EVENT_SEQ_3"].fillna("None").replace("  ", "None")

df["road_location"] = df["road_location"].fillna("None").replace("  ", "None")
df["ROAD_DESCRIPTION"] = df["ROAD_DESCRIPTION"].replace("  ", "None")
df["ROAD_CONTOUR"] = df["ROAD_CONTOUR"].replace("  ", "None")
df["ROAD_CONDITION"] = df["ROAD_CONDITION"].replace("  ", "None")
df["LIGHT_CONDITION"] = df["LIGHT_CONDITION"].fillna("None").replace("  ", "None")

df["TU1_VEHICLE_TYPE"] = df["TU1_VEHICLE_TYPE"].fillna("None").replace({
    "  ": "None",
    "UNK": "None",
    "0": "None",
    "Motorcycle": "MOTORCYCLE",
})
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].fillna("None").replace({
    "  ": "None",
    "021": "None",
    "unk": "None",
    "91": "None",
    "00": "None",
})
df["TU1_VEHICLE_MOVEMENT"] = df["TU1_VEHICLE_MOVEMENT"].fillna("None").replace({
    "  ": "None",
    "021": "None",
    "18": "None",
    "00": "None",
})
df["TU1_DRIVER_ACTION"] = df["TU1_DRIVER_ACTION"].fillna("None").replace({
    "  ": "None",
    "na": "None",
    "00": "None",
    "17": "None",
})
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].fillna("None").replace({
    "  ": "None",
    " 00": "None",
    " 06": "None",
    "18": "None",
    "17": "None",
    "NO APPARENT": "No Apparent Contributing Factor",
})
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].fillna("None").replace({
    "  ": "None",
    "00": "None",
    "/": "None",
    "-": "None",
    "--": "None",
})

df["TU2_VEHICLE_TYPE"] = df["TU2_VEHICLE_TYPE"].fillna("None").replace({
    "  ": "None",
    "UNK": "None",
    "0": "None",
    "Motorcycle": "MOTORCYCLE",
})
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].fillna("None").replace({
    "  ": "None",
    "05": "None",
    "07": "None",
    "01": "None",
    "03": "None",
    "00": "None",
    "02": "None",
})
df["TU2_VEHICLE_MOVEMENT"] = df["TU2_VEHICLE_MOVEMENT"].fillna("None").replace({
    "  ": "None",
    "00": "None",
    "0": "None",
})
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].fillna("None").replace({
    "  ": "None",
    "00": "None",
    "0": "None",
    "non": "None",
    "none": "None",
})
df["TU2_DRIVER_HUMANCONTRIBFACTOR"] = df["TU2_DRIVER_HUMANCONTRIBFACTOR"].fillna("None").replace("  ", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].fillna("None").replace({
    "  ": "None",
    "00": "None",
    "0": "None",
    "/": "None",
    "--": "None",
    "-": "None",
})

# convert the pedestrian and bicyclist indicators to binary fields
# these might be a data leak
df["pedestrian_ind"] = (df["pedestrian_ind"].fillna(0) > 0).astype(int)
df["bicycle_ind"] = (df["bicycle_ind"].fillna(0) > 0).astype(int)

Trimming whitespace...
Filling in missing values...


## Split the time field into separate fields for month, day, hour

In [11]:
# It is necessary to separate the time of the incident 
# into multiple categories.
print("Extracting time features...")

df["first_occurrence_date"] = pd.to_datetime(
    df["first_occurrence_date"], errors="coerce"
)

df["hour"] = df["first_occurrence_date"].dt.hour
df["day_of_week"] = df["first_occurrence_date"].dt.dayofweek   # 0=Mon 6=Sun
df["month"] = df["first_occurrence_date"].dt.month

# We no longer need the raw datetime column
df.drop(columns=["first_occurrence_date"], inplace=True)
list(df.columns)

Extracting time features...


['offense_code',
 'district_id',
 'precinct_id',
 'neighborhood_id',
 'bicycle_ind',
 'pedestrian_ind',
 'HARMFUL_EVENT_SEQ_1',
 'HARMFUL_EVENT_SEQ_2',
 'HARMFUL_EVENT_SEQ_3',
 'road_location',
 'ROAD_DESCRIPTION',
 'ROAD_CONTOUR',
 'ROAD_CONDITION',
 'LIGHT_CONDITION',
 'TU1_VEHICLE_TYPE',
 'TU1_TRAVEL_DIRECTION',
 'TU1_VEHICLE_MOVEMENT',
 'TU1_DRIVER_ACTION',
 'TU1_DRIVER_HUMANCONTRIBFACTOR',
 'TU1_PEDESTRIAN_ACTION',
 'TU2_VEHICLE_TYPE',
 'TU2_TRAVEL_DIRECTION',
 'TU2_VEHICLE_MOVEMENT',
 'TU2_DRIVER_ACTION',
 'TU2_DRIVER_HUMANCONTRIBFACTOR',
 'TU2_PEDESTRIAN_ACTION',
 'high_risk',
 'hour',
 'day_of_week',
 'month']

## Save the cleaned dataframe to a new CSV file

In [12]:
df.to_csv("data/processed_crime_traffic_accidents.csv", index=False)
# I'm finding it necessary to reload the data to make the classes in the categories consistent in terms of data types
df = pd.read_csv("data/processed_crime_traffic_accidents.csv")

## Force all data to be strings
There aren't any numeric fields, so we can just convert everything to strings to make it easier to one-hot encode later on.

In [17]:
# force all columns to be strings (except the target variable)
for col in df.columns:
    if col != "high_risk":
        df[col] = df[col].astype(str)

## Train / Test Split

In [14]:
X = df.drop(columns=["high_risk"])
y = df["high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=1234,
    stratify=y,      # keep class balance in both splits
)

weights = np.where(y_train == 1, 10, 1)

## One-hot encode the categorical features

In [15]:
# all the data is categorical, one-hot encode everything

encoder = OneHotEncoder(handle_unknown="ignore")
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)
X_train_encoded.shape

(225795, 976)

## Random Forest

In [ ]:
# # train the model
# model = RandomForestClassifier(
#     random_state=1234,
#     n_estimators=1000,
#     min_samples_leaf=5,
#     class_weight='balanced',
#     bootstrap=True,
#     oob_score=True,
#     n_jobs=-1
# )
# model.fit(X_train_encoded, y_train)


# # evalute model performance on test set

# # Get the depth of every tree in the forest
# depths = [estimator.get_depth() for estimator in model.estimators_]
# print(f"Average depth: {sum(depths)/len(depths)}")
# print(f"Max depth: {max(depths)}")
# print(f"OOB Score: {model.oob_score_}")

# y_pred_test = model.predict(X_test_encoded)

# print(classification_report(y_test, y_pred_test, digits=4))

# precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)
# print(f"Precision: {precision}")
# print(f"Recall:    {recall}")
# print(f"F1 Score:  {f1}")


# # pickle the model
# os.makedirs("pickle_jar", exist_ok=True)
# with open(f"./pickle_jar/model_{f1:.4f}.pkl", "wb") as f:
#     pickle.dump(model, f) 

## Random Forest Grid Search

In [ ]:
print("Performing Random Forest Grid Search...")

# make sure the classes are balanced
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)

# build the grid search
param_grid = {
    # Number of trees in the forest. 
    # More is usually better but takes longer to train on a local machine.
    'n_estimators': [500, 1000],
    # The number of features to consider when looking for the best split.
    # 'sqrt' is ~31 features, 'log2' is ~10 features.
    'max_features': ['sqrt', 'log2'],
    # Maximum number of levels in each tree. 
    # Setting this to None allows trees to grow until they are pure.
    'max_depth': [None, 10, 20],
    # Minimum number of samples required to be at a leaf node.
    # Increasing this prevents the model from picking up on tiny, noisy patterns.
    'min_samples_leaf': [1, 5, 10],
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', oob_score=True, random_state=1234),
    param_grid=param_grid,
    cv=skf,
    n_jobs=-1,
    scoring='f1', # Since our classes are imbalanced, don't use 'accuracy'
    refit=True,
    verbose=3
)

# find the best parameters
grid_search.fit(X_train_encoded, y_train)
print("Best Random Forest CV Score:", grid_search.best_score_)
print("Best Random Forest Parameters:", grid_search.best_params_)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/rf_model_{grid_search.best_score_:.4f}.pkl", "wb") as f:
    pickle.dump(grid_search.best_estimator_, f)

# save the winning model to a variable
rf_model = grid_search.best_estimator_
y_pred_test = rf_model.predict(X_test_encoded)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

Performing Random Forest Grid Search...
Fitting 3 folds for each of 36 candidates, totalling 108 fits


[CV 2/3] END max_depth=None, max_features=sqrt, min_samples_leaf=5, n_estimators=500;, score=0.349 total time=18.2min
[CV 3/3] END max_depth=None, max_features=sqrt, min_samples_leaf=5, n_estimators=500;, score=0.370 total time=18.2min
[CV 1/3] END max_depth=None, max_features=sqrt, min_samples_leaf=5, n_estimators=500;, score=0.353 total time=18.3min
[CV 1/3] END max_depth=None, max_features=sqrt, min_samples_leaf=1, n_estimators=500;, score=0.074 total time=26.4min
[CV 2/3] END max_depth=None, max_features=sqrt, min_samples_leaf=1, n_estimators=500;, score=0.068 total time=26.5min
[CV 3/3] END max_depth=None, max_features=sqrt, min_samples_leaf=1, n_estimators=500;, score=0.062 total time=26.9min
[CV 2/3] END max_depth=None, max_features=sqrt, min_samples_leaf=10, n_estimators=500;, score=0.305 total time=14.9min
[CV 3/3] END max_depth=None, max_features=sqrt, min_samples_leaf=10, n_estimators=500;, score=0.311 total time=14.8min
[CV 1/3] END max_depth=None, max_features=sqrt, min_sa

/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/3] END max_depth=None, max_features=log2, min_samples_leaf=1, n_estimators=500;, score=0.012 total time=184.2min
[CV 2/3] END max_depth=None, max_features=sqrt, min_samples_leaf=10, n_estimators=1000;, score=0.305 total time=193.1min
[CV 1/3] END max_depth=None, max_features=sqrt, min_samples_leaf=10, n_estimators=1000;, score=0.301 total time=193.2min
[CV 3/3] END max_depth=None, max_features=sqrt, min_samples_leaf=10, n_estimators=1000;, score=0.312 total time=193.4min
[CV 1/3] END max_depth=None, max_features=log2, min_samples_leaf=5, n_estimators=500;, score=0.247 total time= 5.8min
[CV 2/3] END max_depth=None, max_features=log2, min_samples_leaf=5, n_estimators=500;, score=0.244 total time= 5.6min
[CV 3/3] END max_depth=None, max_features=log2, min_samples_leaf=5, n_estimators=500;, score=0.247 total time= 5.7min
[CV 2/3] END max_depth=None, max_features=log2, min_samples_leaf=10, n_estimators=500;, score=0.207 total time= 4.2min
[CV 1/3] END max_depth=None, max_features=log

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

# train the model
model = ExtraTreesClassifier(
    random_state=1234,
    n_estimators=500,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1
)
model.fit(X_train_encoded, y_train)


# evalute model performance on test set

# Get the depth of every tree in the forest
depths = [estimator.get_depth() for estimator in model.estimators_]
print(f"Average depth: {sum(depths)/len(depths)}")
print(f"Max depth: {max(depths)}")

y_pred_test = model.predict(X_test_encoded)

print(classification_report(y_test, y_pred_test, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"F1 Score:  {f1}")


# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/et_model_wbalanced_{f1[1]:.4f}.pkl", "wb") as f:
    pickle.dump(model, f) 

Average depth: 83.99
Max depth: 129
              precision    recall  f1-score   support

           0     0.9878    0.9686    0.9781     55137
           1     0.2738    0.4970    0.3531      1312

    accuracy                         0.9577     56449
   macro avg     0.6308    0.7328    0.6656     56449
weighted avg     0.9712    0.9577    0.9636     56449

Precision: [0.98779315 0.27383452]
Recall:    [0.96864175 0.49695122]
F1 Score:  [0.97812371 0.35310046]


## AdaBoost Grid Search

In [ ]:
print("Performing AdaBoost Grid Search...")

# make sure the classes are balanced
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)

# build the grid search
param_grid = {
    'n_estimators': [1000, 2000],
    'learning_rate': [0.1, 0.5, 1.0],
    'estimator__max_depth': [1, 2, 3] # Accessing the base tree parameters
}
ada_grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=1234), random_state=1234),
    param_grid=param_grid,
    scoring='f1', # Crucial for our fatality predictions
    cv=skf,
    n_jobs=-1,
    refit=True,
    verbose=3
)

# find the best parameters
ada_grid_search.fit(X_train_encoded, y_train)

# print some of the results:
print("Best AdaBoost CV F1 Score:", ada_grid_search.best_score_)
print("Best AdaBoost Params:", ada_grid_search.best_params_)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/ada_model_{ada_grid_search.best_score_:.4f}.pkl", "wb") as f:
    pickle.dump(ada_grid_search.best_estimator_, f)

# save the winning model to a variable
ada_model = ada_grid_search.best_estimator_
y_pred_test = ada_model.predict(X_test_encoded)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

Performing AdaBoost Grid Search...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


KeyboardInterrupt: 

## Linear SVC

In [ ]:
# make sure the classes are balanced
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)


# build the grid search
param_grid = {
    'C': [0.001, 0.01, 0.1, 1],
    'loss': ['hinge', 'squared_hinge']
}
svc_grid = GridSearchCV(
    estimator=LinearSVC(class_weight='balanced', dual="auto", random_state=1234, max_iter=50000),
    param_grid=param_grid,
    cv=skf,
    scoring='f1', # Still focusing on that balance of Precision and Recall
    n_jobs=-1,
    refit=True,
    verbose=3
)

# find best parameters with grid search
svc_grid.fit(X_train_encoded, y_train, sample_weight=weights)

# print some of the results:
print("Best SVC CV F1 Score:", svc_grid.best_score_)
print("Best SVC Params:", svc_grid.best_params_)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/svc_model_w25_{svc_grid.best_score_:.4f}.pkl", "wb") as f:
    pickle.dump(svc_grid.best_estimator_, f)

# save the winning model to a variable
svc_model = svc_grid.best_estimator_
y_pred_test = svc_model.predict(X_test_encoded)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV 1/3] END ...............C=0.001, loss=hinge;, score=0.629 total time=   1.4s
[CV 2/3] END ...............C=0.001, loss=hinge;, score=0.636 total time=   1.6s
[CV 3/3] END ...............C=0.001, loss=hinge;, score=0.628 total time=   1.4s
[CV 3/3] END ................C=0.01, loss=hinge;, score=0.640 total time=   2.8s
[CV 2/3] END ................C=0.01, loss=hinge;, score=0.648 total time=   2.6s
[CV 1/3] END ................C=0.01, loss=hinge;, score=0.641 total time=   3.4s
[CV 1/3] END .......C=0.001, loss=squared_hinge;, score=0.644 total time=   5.3s
[CV 3/3] END .......C=0.001, loss=squared_hinge;, score=0.642 total time=   5.3s
[CV 2/3] END .......C=0.001, loss=squared_hinge;, score=0.649 total time=   5.7s
[CV 2/3] END ........C=0.01, loss=squared_hinge;, score=0.649 total time=  11.0s
[CV 1/3] END ........C=0.01, loss=squared_hinge;, score=0.648 total time=  11.2s
[CV 3/3] END ........C=0.01, loss=squared_hinge;,

/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 3/3] END ...................C=1, loss=hinge;, score=0.639 total time=  56.9s


/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 1/3] END ...................C=1, loss=hinge;, score=0.640 total time= 1.0min
[CV 2/3] END ...................C=1, loss=hinge;, score=0.637 total time= 1.0min
Best SVC CV F1 Score: 0.6463172978088911
Best SVC Params: {'C': 0.01, 'loss': 'squared_hinge'}
              precision    recall  f1-score   support

           0     0.9940    0.8440    0.9128     55137
           1     0.1068    0.7843    0.1880      1312

    accuracy                         0.8426     56449
   macro avg     0.5504    0.8141    0.5504     56449
weighted avg     0.9733    0.8426    0.8960     56449



## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Initialize the model
# 'class_weight=balanced' handles the rare fatalities
# 'solver=liblinear' is highly optimized for large, sparse datasets (1,000+ columns)
# 'max_iter=2000' prevents annoying convergence warnings
log_reg = LogisticRegression(
    class_weight='balanced', 
    # solver='liblinear', 
    random_state=1234, 
    max_iter=2000,
    # l1_ratio=1
)

# 2. Define the grid
param_grid = {
    # L1 acts as a feature selector (it will delete useless columns by setting weight to 0)
    # L2 shrinks the weights of less important columns but keeps them all
    # 'penalty': ['l1', 'l2'],
    'l1_ratio': [0, 1],  # Only used if penalty='elasticnet'
    
    # C is the inverse of regularization strength (just like in the SVM)
    # Smaller values (0.01) = Stronger penalty (simpler model, prevents overfitting)
    # Larger values (10) = Weaker penalty (more complex model)
    'C': [0.001, 0.01, 0.1],
    "solver": ['liblinear', 'lbfgs', 'newton-cholesky', 'sag', 'saga']
}

# 3. Setup Stratified CV
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)

# 4. Run Grid Search
grid_log_reg = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',  # Optimize for finding fatalities
    n_jobs=-1,     # Use all CPU cores
    verbose=3      # Print detailed progress
)

print("Starting Logistic Regression Grid Search...")
grid_log_reg.fit(X_train_encoded, y_train, sample_weight=weights)

# 5. Output the results
print("Best Parameters:", grid_log_reg.best_params_)
print("Best CV F1 Score:", grid_log_reg.best_score_)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/log_reg_model_w25_{grid_log_reg.best_score_:.4f}.pkl", "wb") as f:
    pickle.dump(grid_log_reg.best_estimator_, f)

# 6. Save the winning model
lr_model = grid_log_reg.best_estimator_
y_pred_test = lr_model.predict(X_test_encoded)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

Starting Logistic Regression Grid Search...
Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV 2/3] END .C=0.001, l1_ratio=0, solver=lbfgs;, score=0.621 total time=   1.5s
[CV 1/3] END .C=0.001, l1_ratio=0, solver=lbfgs;, score=0.621 total time=   1.7s
[CV 3/3] END .C=0.001, l1_ratio=0, solver=lbfgs;, score=0.619 total time=   2.0s
[CV 2/3] END C=0.001, l1_ratio=0, solver=liblinear;, score=0.621 total time=   2.5s
[CV 1/3] END C=0.001, l1_ratio=0, solver=liblinear;, score=0.621 total time=   2.6s
[CV 3/3] END C=0.001, l1_ratio=0, solver=liblinear;, score=0.620 total time=   3.3s
[CV 1/3] END ...C=0.001, l1_ratio=0, solver=sag;, score=0.621 total time=   4.5s
[CV 1/3] END ...C=0.001, l1_ratio=1, solver=lbfgs;, score=nan total time=   0.1s
[CV 2/3] END ...C=0.001, l1_ratio=0, solver=sag;, score=0.621 total time=   4.6s
[CV 2/3] END ...C=0.001, l1_ratio=1, solver=lbfgs;, score=nan total time=   0.1s
[CV 3/3] END ...C=0.001, l1_ratio=0, solver=sag;, score=0.619 total time=   

/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 1/3] END ..C=0.001, l1_ratio=0, solver=saga;, score=0.346 total time= 5.6min


/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 3/3] END ..C=0.001, l1_ratio=0, solver=saga;, score=0.441 total time= 5.7min


...

[CV 2/3] END ....C=0.1, l1_ratio=1, solver=saga;, score=0.568 total time=12.9min


/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 1/3] END ....C=0.1, l1_ratio=1, solver=saga;, score=0.533 total time=13.0min


/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
27 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    es

[CV 3/3] END ....C=0.1, l1_ratio=1, solver=saga;, score=0.543 total time=13.0min
Best Parameters: {'C': 0.1, 'l1_ratio': 1, 'solver': 'liblinear'}
Best CV F1 Score: 0.6447853200642287
              precision    recall  f1-score   support

           0     0.9939    0.8432    0.9124     55137
           1     0.1063    0.7835    0.1871      1312

    accuracy                         0.8418     56449
   macro avg     0.5501    0.8134    0.5498     56449
weighted avg     0.9733    0.8418    0.8955     56449



## LightGBM

In [ ]:
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# 1. Setup the model
lgbm = LGBMClassifier(
    objective='binary',
    is_unbalance=True,
    verbose=-1,
    n_jobs=-1,
    callbacks=[early_stopping(stopping_rounds=50)]
)

# 2. Define DISTRIBUTIONS instead of fixed lists
# randint(a, b) picks a random integer between a and b
# uniform(a, b) picks a random float between a and a+b
param_dist = {
    'n_estimators': randint(1000, 5000),
    'learning_rate': uniform(0.01, 0.1),
    'num_leaves': randint(5, 500),
    'feature_fraction': uniform(0.5, 0.5),
    'min_child_samples': randint(10, 100)
}

# 3. Setup the Randomized Search
lgbm_random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=10,           # TOTAL combinations to try (The "Budget")
    cv=3,                # Still using 3-fold Stratified CV
    scoring='f1',
    verbose=3,
    random_state=1234,
    n_jobs=-1
)

# 4. Run it
# NOTE: this model it is better to not use one-hot encoded data
lgbm_X_train = X_train.copy()
lgbm_X_test = X_test.copy()
for col in lgbm_X_train.columns:
    lgbm_X_train[col] = lgbm_X_train[col].astype('category')
    lgbm_X_test[col] = lgbm_X_test[col].astype('category')
lgbm_random_search.fit(lgbm_X_train, y_train)

# 5. Access Results
print(f"Best Params: {lgbm_random_search.best_params_}")
print(f"Best CV F1 Score: {lgbm_random_search.best_score_}")

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/lgbm_model_{lgbm_random_search.best_score_:.4f}.pkl", "wb") as f:
    pickle.dump(lgbm_random_search.best_estimator_, f)

# 6. Save the winning model
lgbm_model = lgbm_random_search.best_estimator_
y_pred_test = lgbm_model.predict(lgbm_X_test)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV 1/3] END feature_fraction=0.8856799594312005, learning_rate=0.09606697713776872, min_child_samples=36, n_estimators=1030, num_leaves=176;, score=0.287 total time= 2.4min
[CV 3/3] END feature_fraction=0.8856799594312005, learning_rate=0.09606697713776872, min_child_samples=36, n_estimators=1030, num_leaves=176;, score=0.303 total time= 2.4min
[CV 2/3] END feature_fraction=0.8856799594312005, learning_rate=0.09606697713776872, min_child_samples=36, n_estimators=1030, num_leaves=176;, score=0.294 total time= 2.5min
[CV 2/3] END feature_fraction=0.5957597251894462, learning_rate=0.07221087710398319, min_child_samples=86, n_estimators=4444, num_leaves=157;, score=0.307 total time= 7.9min
[CV 1/3] END feature_fraction=0.5957597251894462, learning_rate=0.07221087710398319, min_child_samples=86, n_estimators=4444, num_leaves=157;, score=0.287 total time= 8.0min
[CV 2/3] END feature_fraction=0.7434167216847201, learning_rate=0.043

# Jump in here to evaluate models
## Evaluate models on the test set and compare results

In [16]:
import pickle
from sklearn.metrics import classification_report, confusion_matrix

print(f"Shape of X_train_encoded: {X_train_encoded.shape}")
print(f"Shape of X_test_encoded: {X_test_encoded.shape}")
print()
# extra trees
with open(f"./pickle_jar/et_model_wbalanced_0.3531.pkl", "rb") as f:
    et_model = pickle.load(f)
# retrain the model with the entire test set
et_model.n_jobs = -1
et_model.fit(X_train_encoded, y_train)
y_test_pred = et_model.predict(X_test_encoded)
print("Extra Trees Test Set Performance:")
print(classification_report(y_test, y_test_pred, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print()
print()
# save the final version of the model
with open(f"./pickle_jar/et_model_final.pkl", "wb") as f:
    pickle.dump(et_model, f)


# random forest
with open(f"./pickle_jar/rf_model_wbalanced_0.3572.pkl", "rb") as f:
    rf_model = pickle.load(f)
# retrain the model with the entire test set
rf_model.n_jobs = -1
rf_model.fit(X_train_encoded, y_train)
y_test_pred = rf_model.predict(X_test_encoded)
print("Random Forest Test Set Performance:")
print(classification_report(y_test, y_test_pred, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print()
print()
# save the final version of the model
with open(f"./pickle_jar/rf_model_final.pkl", "wb") as f:
    pickle.dump(rf_model, f)


# logistic regression
with open(f"./pickle_jar/lr_model_w50_0.8291.pkl", "rb") as f:
    lr_model = pickle.load(f)
# retrain the model with the entire test set
lr_model.n_jobs = -1
lr_model.fit(X_train_encoded, y_train)
y_pred_test = lr_model.predict(X_test_encoded)
print("Logistic Regression Test Set Performance:")
print(classification_report(y_test, y_pred_test, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))
print()
print()
# save the final version of the model
with open(f"./pickle_jar/lr_model_final.pkl", "wb") as f:
    pickle.dump(lr_model, f)


# Linear SVC
with open(f"./pickle_jar/svc_model_w50_0.8279.pkl", "rb") as f:
    svc_model = pickle.load(f)
# retrain the model with the entire test set
svc_model.n_jobs = -1
svc_model.fit(X_train_encoded, y_train)
y_pred_test = svc_model.predict(X_test_encoded)
print("Linear SVC Test Set Performance:")
print(classification_report(y_test, y_pred_test, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))
print()
print()
# save the final version of the model
with open(f"./pickle_jar/svc_model_final.pkl", "wb") as f:
    pickle.dump(svc_model, f)



# LightGBM
with open(f"./pickle_jar/lgbm_model_0.3621.pkl", "rb") as f:
    lgbm_model = pickle.load(f)
# retrain the model with the entire test set
lgbm_model.n_jobs = -1
lgbm_X_train = X_train.copy()
lgbm_X_test = X_test.copy()
for col in lgbm_X_test.columns:
    lgbm_X_train[col] = lgbm_X_train[col].astype('category')
    lgbm_X_test[col] = lgbm_X_test[col].astype('category')
lgbm_model.fit(lgbm_X_train, y_train)
y_pred_test = lgbm_model.predict(lgbm_X_test)
print("LightGBM Test Set Performance:")
print(classification_report(y_test, y_pred_test, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))
# save the final version of the model
with open(f"./pickle_jar/lgbm_model_final.pkl", "wb") as f:
    pickle.dump(lgbm_model, f)

Shape of X_train_encoded: (225795, 976)
Shape of X_test_encoded: (56449, 976)

Extra Trees Test Set Performance:
              precision    recall  f1-score   support

           0     0.9878    0.9688    0.9782     55137
           1     0.2739    0.4954    0.3528      1312

    accuracy                         0.9577     56449
   macro avg     0.6308    0.7321    0.6655     56449
weighted avg     0.9712    0.9577    0.9636     56449

Confusion Matrix:
[[53414  1723]
 [  662   650]]


Random Forest Test Set Performance:
              precision    recall  f1-score   support

           0     0.9872    0.9726    0.9799     55137
           1     0.2906    0.4710    0.3594      1312

    accuracy                         0.9610     56449
   macro avg     0.6389    0.7218    0.6696     56449
weighted avg     0.9710    0.9610    0.9655     56449

Confusion Matrix:
[[53628  1509]
 [  694   618]]




/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression Test Set Performance:
              precision    recall  f1-score   support

           0     0.9941    0.8383    0.9095     55137
           1     0.1042    0.7904    0.1841      1312

    accuracy                         0.8371     56449
   macro avg     0.5491    0.8143    0.5468     56449
weighted avg     0.9734    0.8371    0.8927     56449

Confusion Matrix:
[[46219  8918]
 [  275  1037]]


Linear SVC Test Set Performance:
              precision    recall  f1-score   support

           0     0.9940    0.8389    0.9099     55137
           1     0.1042    0.7881    0.1841      1312

    accuracy                         0.8377     56449
   macro avg     0.5491    0.8135    0.5470     56449
weighted avg     0.9733    0.8377    0.8930     56449

Confusion Matrix:
[[46252  8885]
 [  278  1034]]


LightGBM Test Set Performance:
              precision    recall  f1-score   support

           0     0.9884    0.9661    0.9771     55137
           1     0.2692    0.